In [20]:
# 4팀 전달용 Output 생성에 필요한 라이브러리
import json
import pandas as pd

from typing import Any

In [21]:
# Priority 결과(JSON 파일)를 DataFrame으로 변환
def load_output_input(json_path):

    df = pd.read_json(json_path)

    if df.empty:
        raise ValueError("Priority 결과 JSON이 비어 있습니다.")

    return df


# -------------------------------
# 테스트용
#output_df = load_output_input("priority_output.json")

# 실제 연동 시에도 동일
output_df = load_output_input("priority.json")
# -------------------------------

print("Output 입력 데이터 크기:", output_df.shape)

Output 입력 데이터 크기: (121, 15)


In [22]:
# Priority 결과에 필요한 필수 컬럼 검증
required_columns = [
    "priority_rank",
    "cve_id",
    "cvss_score",
    "predicted_severity",
    "priority_score",
    "response_priority",
    "description",
]

missing_columns = [
    column
    for column in required_columns
    if column not in output_df.columns
]

if missing_columns:
    raise ValueError(
        f"Priority 결과에 필요한 컬럼이 없습니다: {missing_columns}"
    )

print("Priority 결과 필수 컬럼 확인 완료")

Priority 결과 필수 컬럼 확인 완료


In [23]:
# 4팀에서 사용할 최종 컬럼 구성
team4_columns = [
    "priority_rank",
    "cve_id",
    "service",
    "version",
    "cwe",
    "cvss_score",
    "severity",
    "predicted_severity",
    "prediction_confidence",
    "priority_score",
    "response_priority",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "description",
]

# 실제 존재하는 컬럼만 선택
team4_columns = [
    column
    for column in team4_columns
    if column in output_df.columns
]

team4_df = output_df[
    team4_columns
].copy()

print("4팀 전달용 컬럼 정리 완료")
print("전달 컬럼:", team4_df.columns.tolist())

display(team4_df.head())

4팀 전달용 컬럼 정리 완료
전달 컬럼: ['priority_rank', 'cve_id', 'service', 'version', 'cwe', 'cvss_score', 'severity', 'predicted_severity', 'priority_score', 'response_priority', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'description']


,priority_rank,cve_id,service,version,cwe,cvss_score,severity,predicted_severity,priority_score,response_priority,attack_vector,attack_complexity,privileges_required,user_interaction,description
0,1,CVE-2010-4478,ssh,4.7p1 Debian 8ubuntu1,CWE-287,9.8,CRITICAL,CRITICAL,99.0,긴급 대응,NETWORK,LOW,NONE,NONE,"OpenSSH 5.6 and earlier, when J-PAKE is enable..."
1,2,CVE-2009-3555,http,2.2.8,CWE-295,9.8,CRITICAL,CRITICAL,99.0,긴급 대응,NETWORK,LOW,NONE,NONE,"The TLS protocol, and the SSL protocol 3.0 and..."
2,3,CVE-2010-0425,http,2.2.8,UNKNOWN,10.0,HIGH,HIGH,90.0,긴급 대응,NETWORK,LOW,NONE,UNKNOWN,modules/arch/win32/mod_isapi.c in mod_isapi in...
3,4,CVE-2011-2523,ftp,2.3.4,CWE-78,9.8,CRITICAL,MEDIUM,89.0,긴급 대응,NETWORK,LOW,NONE,NONE,vsftpd 2.3.4 downloaded between 20110630 and 2...
4,5,CVE-2016-1908,ssh,4.7p1 Debian 8ubuntu1,CWE-287,9.8,CRITICAL,MEDIUM,89.0,긴급 대응,NETWORK,LOW,NONE,NONE,The client in OpenSSH before 7.2 mishandles fa...


In [24]:
# 선택한 취약점 정보를 생성형 AI에 전달할 문장으로 구성
def build_chatbot_context(
    record: dict[str, Any],
) -> str:

    return f"""
다음 취약점 정보를 분석해 주세요.

CVE ID: {record.get("cve_id", "정보 없음")}
서비스: {record.get("service", "정보 없음")}
버전: {record.get("version", "정보 없음")}
CWE: {record.get("cwe", "정보 없음")}

CVSS Score: {record.get("cvss_score", "정보 없음")}
기존 Severity: {record.get("severity", "정보 없음")}
모델 예측 Severity: {record.get("predicted_severity", "정보 없음")}
예측 신뢰도: {record.get("prediction_confidence", "정보 없음")}

Priority Score: {record.get("priority_score", "정보 없음")}
대응 우선순위: {record.get("response_priority", "정보 없음")}

Attack Vector: {record.get("attack_vector", "정보 없음")}
Attack Complexity: {record.get("attack_complexity", "정보 없음")}
Privileges Required: {record.get("privileges_required", "정보 없음")}
User Interaction: {record.get("user_interaction", "정보 없음")}

Description:
{record.get("description", "정보 없음")}
""".strip()

In [25]:
# AI 분석 결과를 취약점 정보에 결합
def merge_ai_result(
    record: dict[str, Any],
    ai_result: dict[str, str],
) -> dict[str, Any]:

    enriched_record = record.copy()

    enriched_record["ai_summary"] = ai_result.get(
        "summary",
        "",
    )

    enriched_record["risk_reason"] = ai_result.get(
        "risk_reason",
        "",
    )

    enriched_record["recommended_action"] = ai_result.get(
        "recommended_action",
        "",
    )

    enriched_record["response_plan"] = ai_result.get(
        "response_plan",
        "",
    )

    enriched_record["verification_method"] = ai_result.get(
        "verification_method",
        "",
    )

    return enriched_record

In [26]:
# OpenAI API 연결 전 구조 확인용 임시 AI 결과
mock_ai_result = {
    "summary": (
        "외부 네트워크를 통해 악용될 가능성이 있는 "
        "고위험 취약점입니다."
    ),
    "risk_reason": (
        "공격 복잡도가 낮고 인증 없이 접근할 수 있어 "
        "악용 가능성이 높습니다."
    ),
    "recommended_action": (
        "영향받는 버전을 확인한 후 보안 패치를 우선 적용하고 "
        "외부 접근을 제한해야 합니다."
    ),
    "response_plan": (
        "영향 자산 식별 → 임시 접근 통제 → 패치 적용 → "
        "서비스 점검 → 재스캔 순서로 대응합니다."
    ),
    "verification_method": (
        "패치 적용 후 버전을 확인하고 동일 조건으로 재스캔하여 "
        "취약점이 더 이상 탐지되지 않는지 검증합니다."
    ),
}

# 우선순위 1위 취약점으로 테스트
test_record = team4_df.iloc[0].to_dict()

chatbot_context = build_chatbot_context(
    test_record
)

test_output = merge_ai_result(
    test_record,
    mock_ai_result,
)

print("챗봇 입력 Context")
print(chatbot_context)

print("\nAI 결과 결합 테스트")
print(
    json.dumps(
        test_output,
        ensure_ascii=False,
        indent=2,
    )
)

챗봇 입력 Context
다음 취약점 정보를 분석해 주세요.

CVE ID: CVE-2010-4478
서비스: ssh
버전: 4.7p1 Debian 8ubuntu1
CWE: CWE-287

CVSS Score: 9.8
기존 Severity: CRITICAL
모델 예측 Severity: CRITICAL
예측 신뢰도: 정보 없음

Priority Score: 99.0
대응 우선순위: 긴급 대응

Attack Vector: NETWORK
Attack Complexity: LOW
Privileges Required: NONE
User Interaction: NONE

Description:
OpenSSH 5.6 and earlier, when J-PAKE is enabled, does not properly validate the public parameters in the J-PAKE protocol, which allows remote attackers to bypass the need for knowledge of the shared secret, and successfully authenticate, by sending crafted values in each round of the protocol, a related issue to CVE-2010-4252.

AI 결과 결합 테스트
{
  "priority_rank": 1,
  "cve_id": "CVE-2010-4478",
  "service": "ssh",
  "version": "4.7p1 Debian 8ubuntu1",
  "cwe": "CWE-287",
  "cvss_score": 9.8,
  "severity": "CRITICAL",
  "predicted_severity": "CRITICAL",
  "priority_score": 99.0,
  "response_priority": "긴급 대응",
  "attack_vector": "NETWORK",
  "attack_complexity": "L

In [27]:
def analyze_vulnerability(
    record: dict[str, Any],
) -> dict[str, str]:

    context = build_chatbot_context(
        record
    )

    # 여기에서 OpenAI Responses API 호출
    # 응답을 아래 구조의 dict로 변환

    return {
        "summary": "...",
        "risk_reason": "...",
        "recommended_action": "...",
        "response_plan": "...",
        "verification_method": "...",
    }

In [28]:
# 4팀 Streamlit 전달용 전체 결과
team4_output = team4_df.to_dict(
    orient="records"
)

print("4팀 전달용 Output 생성 완료")
print("결과 개수:", len(team4_output))
print("전달 형식:", type(team4_output))

4팀 전달용 Output 생성 완료
결과 개수: 121
전달 형식: <class 'list'>


In [29]:
# 4팀 전달용 JSON 저장
with open(
    "team4_output.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        team4_output,
        file,
        ensure_ascii=False,
        indent=4,
    )

# 확인 및 보고서용 CSV 저장
team4_df.to_csv(
    "team4_output.csv",
    index=False,
    encoding="utf-8-sig",
)

print("team4_output.json 생성 완료")
print("team4_output.csv 생성 완료")

team4_output.json 생성 완료
team4_output.csv 생성 완료


In [30]:
# 최종 전달 결과 확인
print("=" * 60)
print("팀3 → 팀4 최종 전달 결과")
print("=" * 60)

print("전달 형식: JSON")
print("전체 취약점 수:", len(team4_df))

print("\n예측 위험도 분포")
print(
    team4_df["predicted_severity"].value_counts()
)

print("\n대응 우선순위 분포")
print(
    team4_df["response_priority"].value_counts()
)

print("\n우선순위 TOP 10")
display(
    team4_df.head(10)
)
# 팀4 전달용 JSON 저장
team4_df.to_json(
    "team4_output.json",
    orient="records",
    indent=4,
    force_ascii=False,
)

print("\nteam4_output.json 저장 완료")

팀3 → 팀4 최종 전달 결과
전달 형식: JSON
전체 취약점 수: 121

예측 위험도 분포
predicted_severity
MEDIUM      70
HIGH        30
LOW         19
CRITICAL     2
Name: count, dtype: int64

대응 우선순위 분포
response_priority
검토 필요    45
일반 관리    42
우선 대응    27
긴급 대응     7
Name: count, dtype: int64

우선순위 TOP 10


,priority_rank,cve_id,service,version,cwe,cvss_score,severity,predicted_severity,priority_score,response_priority,attack_vector,attack_complexity,privileges_required,user_interaction,description
0,1,CVE-2010-4478,ssh,4.7p1 Debian 8ubuntu1,CWE-287,9.8,CRITICAL,CRITICAL,99.0,긴급 대응,NETWORK,LOW,NONE,NONE,"OpenSSH 5.6 and earlier, when J-PAKE is enable..."
1,2,CVE-2009-3555,http,2.2.8,CWE-295,9.8,CRITICAL,CRITICAL,99.0,긴급 대응,NETWORK,LOW,NONE,NONE,"The TLS protocol, and the SSL protocol 3.0 and..."
2,3,CVE-2010-0425,http,2.2.8,UNKNOWN,10.0,HIGH,HIGH,90.0,긴급 대응,NETWORK,LOW,NONE,UNKNOWN,modules/arch/win32/mod_isapi.c in mod_isapi in...
3,4,CVE-2011-2523,ftp,2.3.4,CWE-78,9.8,CRITICAL,MEDIUM,89.0,긴급 대응,NETWORK,LOW,NONE,NONE,vsftpd 2.3.4 downloaded between 20110630 and 2...
4,5,CVE-2016-1908,ssh,4.7p1 Debian 8ubuntu1,CWE-287,9.8,CRITICAL,MEDIUM,89.0,긴급 대응,NETWORK,LOW,NONE,NONE,The client in OpenSSH before 7.2 mishandles fa...
5,6,CVE-2016-1286,domain,9.4.2,UNKNOWN,8.6,HIGH,HIGH,88.0,긴급 대응,NETWORK,LOW,NONE,NONE,named in ISC BIND 9.x before 9.9.8-P4 and 9.10...
6,7,CVE-2008-0122,domain,9.4.2,CWE-189,10.0,HIGH,MEDIUM,85.0,긴급 대응,NETWORK,LOW,NONE,UNKNOWN,Off-by-one error in the inet_network function ...
7,8,CVE-2012-1667,domain,9.4.2,CWE-189,8.5,HIGH,HIGH,82.5,우선 대응,NETWORK,LOW,NONE,UNKNOWN,"ISC BIND 9.x before 9.7.6-P1, 9.8.x before 9.8..."
8,9,CVE-2010-5107,ssh,4.7p1 Debian 8ubuntu1,CWE-400,7.5,HIGH,HIGH,82.5,우선 대응,NETWORK,LOW,NONE,NONE,The default configuration of OpenSSH through 6...
9,10,CVE-2016-10708,ssh,4.7p1 Debian 8ubuntu1,CWE-476,7.5,HIGH,HIGH,82.5,우선 대응,NETWORK,LOW,NONE,NONE,sshd in OpenSSH before 7.4 allows remote attac...



team4_output.json 저장 완료


In [31]:
print(team4_df.columns)

print(team4_df.head())

print(team4_df.shape)

Index(['priority_rank', 'cve_id', 'service', 'version', 'cwe', 'cvss_score',
       'severity', 'predicted_severity', 'priority_score', 'response_priority',
       'attack_vector', 'attack_complexity', 'privileges_required',
       'user_interaction', 'description'],
      dtype='str')
   priority_rank         cve_id service                version      cwe  \
0              1  CVE-2010-4478     ssh  4.7p1 Debian 8ubuntu1  CWE-287   
1              2  CVE-2009-3555    http                  2.2.8  CWE-295   
2              3  CVE-2010-0425    http                  2.2.8  UNKNOWN   
3              4  CVE-2011-2523     ftp                  2.3.4   CWE-78   
4              5  CVE-2016-1908     ssh  4.7p1 Debian 8ubuntu1  CWE-287   

   cvss_score  severity predicted_severity  priority_score response_priority  \
0         9.8  CRITICAL           CRITICAL            99.0             긴급 대응   
1         9.8  CRITICAL           CRITICAL            99.0             긴급 대응   
2        10.0      HIG

<핵심 과정>

1. 팀2의 CVE 분석 결과를 list[dict] 형태로 입력받음
2. 입력 데이터를 DataFrame으로 변환하고 필수 컬럼을 검증함
3. 학습 당시 저장한 One-Hot Encoding 컬럼과 TF-IDF Vectorizer를 적용함
4. Random Forest 모델을 이용해 취약점 Severity를 예측함
5. CVSS 및 공격 조건을 반영하여 우선순위 점수를 계산함
6. 우선순위 순서와 대응 등급을 생성함
7. 4팀 전달용 list[dict], JSON, CSV 결과를 생성함

<결과>

팀2에서 전달받은 취약점 정보를 학습 당시와 동일한 Feature 구조로 변환하여 위험도를 예측하였다. 이후 CVSS Score, 예측 Severity, Attack Vector, Attack Complexity, Privileges Required 및 User Interaction을 종합하여 우선순위 점수를 계산하였다.

최종 결과는 priority_rank, predicted_severity, prediction_confidence, priority_score 및 response_priority 등을 포함하는 list[dict] 형태로 구성하였으며, 4팀에서 Streamlit 시각화, 생성형 AI 기반 대응 방안 작성 및 PDF 보고서 생성에 바로 활용할 수 있도록 JSON과 CSV 파일도 함께 생성하였다.